# Notebook 1: Setup Environment - Spark + Nessie

**Objective**: Initialize the working environment and connect to the Nessie data catalog.

## Key Concepts

- **Apache Spark**: Distributed processing engine for big data
- **Project Nessie**: Data catalog with version control (like Git for data)
- **Apache Iceberg**: ACID table format with time travel

## Architecture

```
Spark Application
        │
        ├── spark_catalog (default catalog)
        │
        └── nessie (our versioned catalog)
                │
                ├── bronze (raw data)
                ├── silver (cleaned data)
                └── gold (business aggregates)
```

**Prerequisites**: 
- Nessie running: `docker run -p 19120:19120 ghcr.io/projectnessie/nessie:latest`
- Data uploaded to S3: `python scripts/upload_to_s3.py --all`

---
## INITIALIZATION

In [1]:
# Configure project path
import sys
import os

project_root = os.path.abspath("..")
src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

# Import Spark modules and configuration
from lakehouse.spark_session import get_spark
from lakehouse.settings import AWS_BUCKET, NESSIE_URI

# Create Spark session
spark = get_spark("demo-setup")

print("=" * 60)
print("LAKEHOUSE DEMO - INITIALIZATION")
print("=" * 60)

LAKEHOUSE DEMO - INITIALIZATION


### Configure Nessie main branch

In [2]:
# Configure Nessie main branch
spark.conf.set("spark.sql.catalog.nessie.ref", "main")

# Create namespaces (medallion architecture layers)
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.bronze")
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.silver")
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.gold")

print("Active branch: main")
print("Nessie namespaces: bronze, silver, gold")
print(f"S3 Bucket: {AWS_BUCKET}")

Active branch: main
Nessie namespaces: bronze, silver, gold
S3 Bucket: iceberg-nessie-lakehouse-demo


### Verify available catalogs

In [3]:
print("=== Available Catalogs ===")
spark.sql("SHOW CATALOGS").show(truncate=False)

=== Available Catalogs ===
+-------------+
|catalog      |
+-------------+
|nessie       |
|spark_catalog|
+-------------+



### Verify Nessie namespaces

In [4]:
print("=== Nessie Namespaces ===")
spark.sql("SHOW NAMESPACES IN nessie").show(truncate=False)

=== Nessie Namespaces ===
+---------+
|namespace|
+---------+
|silver   |
|bronze   |
|gold     |
+---------+



### Verify existing tables

In [8]:
print("=== Tables in Bronze ===")
spark.sql("SHOW TABLES IN nessie.bronze").show(truncate=False)

print("=== Tables in Silver ===")
spark.sql("SHOW TABLES IN nessie.silver").show(truncate=False)

print("=== Tables in Gold ===")
spark.sql("SHOW TABLES IN nessie.gold").show(truncate=False)

=== Tables in Bronze ===
+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+

=== Tables in Silver ===
+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+

=== Tables in Gold ===
+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+



### Test S3 connection and verify batch files

In [6]:
print(f"S3 Bucket configured: {AWS_BUCKET}")
print("\n=== Batch files available on S3 ===")

# Test access to batch files
batch_files = [
    "sales_batch_001.csv",
    "sales_batch_002.csv",
    "sales_batch_003.csv"
]

for batch_file in batch_files:
    batch_path = f"s3a://{AWS_BUCKET}/raw/batches/{batch_file}"
    try:
        test_df = spark.read.option("header", "true").csv(batch_path)
        count = test_df.count()
        print(f"  ✅ {batch_file}: {count:,} records")
    except Exception as e:
        print(f"  ❌ {batch_file}: Not accessible - {e}")

print("\nIf files are not accessible, run:")
print("  python scripts/upload_to_s3.py --all")

S3 Bucket configured: iceberg-nessie-lakehouse-demo

=== Batch files available on S3 ===
  ✅ sales_batch_001.csv: 3,364 records
  ✅ sales_batch_002.csv: 3,501 records
  ✅ sales_batch_003.csv: 3,500 records

If files are not accessible, run:
  python scripts/upload_to_s3.py --all


### Display Nessie branches

In [7]:
print("=== Nessie Branches ===")
spark.sql("LIST REFERENCES IN nessie").show(truncate=False)

=== Nessie Branches ===
+-------+----+----------------------------------------------------------------+
|refType|name|hash                                                            |
+-------+----+----------------------------------------------------------------+
|Branch |main|12c3f87b181927380b5a0f7f536d7282f5c88f29f2c54573d3613ef02cacf6a9|
+-------+----+----------------------------------------------------------------+



### Display active branch

In [9]:
print("=== Active Branch ===")
spark.sql("SHOW REFERENCE IN nessie").show(truncate=False)

current_ref = spark.sql("SHOW REFERENCE IN nessie").first()[1]
print(f"We are on branch: {current_ref}")

=== Active Branch ===
+-------+----+----------------------------------------------------------------+
|refType|name|hash                                                            |
+-------+----+----------------------------------------------------------------+
|Branch |main|12c3f87b181927380b5a0f7f536d7282f5c88f29f2c54573d3613ef02cacf6a9|
+-------+----+----------------------------------------------------------------+

We are on branch: main


### Environment summary

In [10]:
print("""
╔═════════════════════════════════════════════════════════╗
║         LAKEHOUSE ENVIRONMENT READY                     ║
╠═════════════════════════════════════════════════════════╣
║                                                         ║
║  Spark Session   : Configured                           ║
║  Nessie Catalog  : Connected                            ║
║  Iceberg Format  : Enabled                              ║
║  S3 Storage      : Configured                           ║
║                                                         ║
║  Namespaces:                                            ║
║    • nessie.bronze  (raw data)                          ║
║    • nessie.silver  (cleaned data)                      ║
║    • nessie.gold    (business aggregates)               ║
║                                                         ║
║  Active branch:                                         ║
║    • main (production)                                  ║
║                                                         ║
╚═════════════════════════════════════════════════════════╝
""")

print("→ Next step: Run '02_ingestion_bronze.ipynb'")


╔═════════════════════════════════════════════════════════╗
║         LAKEHOUSE ENVIRONMENT READY                     ║
╠═════════════════════════════════════════════════════════╣
║                                                         ║
║  Spark Session   : Configured                           ║
║  Nessie Catalog  : Connected                            ║
║  Iceberg Format  : Enabled                              ║
║  S3 Storage      : Configured                           ║
║                                                         ║
║  Namespaces:                                            ║
║    • nessie.bronze  (raw data)                          ║
║    • nessie.silver  (cleaned data)                      ║
║    • nessie.gold    (business aggregates)               ║
║                                                         ║
║  Active branch:                                         ║
║    • main (production)                                  ║
║                                      